# 01. A2 코넥스 선별 — 대상 만들기와 판정 규칙 고정

**이 노트북이 하는 일:** "코넥스 기업을 분석에 넣을지(A2)"를 **추측이 아니라 데이터로** 정하기 위해, (1) 조회해 볼 코넥스 제조업 기업 명단과 비교용 코스닥 기업 명단을 만들고, (2) **조회 결과를 보기 전에 판정 규칙을 적어 둡니다.**

| 항목 | 내용 |
|---|---|
| 입력 | `outputs/01_파일럿/KIND_상장폐지목록_2019_2024_명단대조.csv`, `기업개황_상장사전체.csv` |
| 출력 | `outputs/02_코넥스_선별/코넥스_선별대상_제조업.csv` (명단) |
| DART 호출 | **없음.** 명단만 만듭니다. 조회는 같은 폴더의 `02_코넥스_선별_수집.py`가 하고, 인증키가 필요해서 **사용자가 직접** 실행합니다 |
| 실행 시간 | 약 3초 |
| 먼저 알아야 할 것 | `src/02_KIND_폐지대조` 노트북 (KIND 대조), `src/01_파일럿/02_재무제표_수집.ipynb` (보고서 요청 방식) |

## 이 노트북의 지도
1. 왜 조회해 보나 (쉬운 설명)
2. **판정 규칙 — 결과를 보기 전에 고정**
3. 조회 대상 만들기
4. 대상 확인
5. 호출 수와 실행 방법
6. 한계

---
# 1. 왜 조회해 보나

**A2는 "코넥스 기업을 분석에 넣을까?"입니다.** 코넥스는 작은 중소·벤처기업 전용 시장입니다(일반 상식이며 이번에 따로 확인하지 않았습니다).

걱정은 하나입니다. **우리 분석은 분기 자료가 필요한데, 코넥스 기업은 DART에 분기 보고서가 없을 수 있습니다.** 파일럿 표본의 코넥스 기업 3곳을 보면 2곳은 재무제표가 아예 조회되지 않았고, 1곳(미쥬)은 사업보고서만 해마다 하나씩 있었습니다. 그런데 **3곳으로는 "코넥스는 그렇다"고 말할 수 없습니다.**

그래서 코넥스 기업 100여 곳의 보고서를 **실제로 조회해서** 분기 보고서가 있는 비율을 봅니다. 비교를 위해 코스닥 기업 40곳도 같은 방식으로 조회합니다.

> 이 조회는 **ICR 같은 결과값을 보지 않고** "보고서가 있는가"만 봅니다. 그래서 결과를 보고 표본을 고르는 문제(결과에 맞춘 선택)가 생기지 않습니다.

---
# 2. 판정 규칙 — 결과를 보기 전에 고정 (2026-11-04)

> **이 규칙은 조회 결과를 보기 전에 적었습니다.** 결과를 본 뒤에 바꾸면 "탐색적 변경"으로 표시하고 이유를 적습니다. **아래의 50%, 20곳은 임의로 정한 값**이며 공식·학술 기준이 아닙니다.

| 항목 | 내용 |
|---|---|
| **핵심 질문** | 코넥스 기업은 DART에서 **분기 보고서(1분기·반기·3분기)** 를 얻을 수 있는가 |
| **분모** | 대상 코넥스 기업 중 **2019년 사업보고서**가 조회되는 기업 (적어도 2019년에 존재한 기업만 봅니다. 늦게 상장해서 자료가 없는 기업은 코넥스의 문제가 아니라 상장 시기의 문제라서 뺍니다) |
| **분자** | 그중 **2019년 1분기·반기·3분기 보고서가 모두** 조회되는 기업 |
| **지표 M** | 분자 ÷ 분모 |
| **규칙** | **M ≥ 50% 이면 코넥스 포함, M < 50% 이면 코넥스 제외** |
| 예외 | 분모가 **20곳 미만**이면 규칙을 적용하지 않고 사용자가 판단 |
| 같이 보고하는 것 | ① 같은 방식으로 계산한 코스닥 기준값(판정에는 쓰지 않음) ② 사전기간 필요 보고서 10개가 모두 있는 기업 비율 ③ 동명이라 같은 회사인지 모호한 기업을 빼고 다시 계산한 M (같은 결론인지 확인) |

### 보고서 조회 범위
사전기간(2019Q1~2021Q2)에 필요한 보고서 **10개**입니다: 2019년(1분기·반기·3분기·사업) + 2020년(같음) + 2021년(1분기·반기). 연결 재무제표를 먼저 조회하고 없으면 별도를 조회합니다(기존 수집과 같은 규칙).

### 용어
- **분기 보고서가 "있다"**: DART가 그 기업의 그 보고서에 대해 정상 응답(상태 000)을 주는 것입니다.
- **2019년 사업보고서만 보는 이유**: 분기 보고서는 사업보고서가 있는 기업에서만 의미가 있기 때문입니다.

In [1]:
import random
import sys
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parents[1]                         # Corp 폴더 (이 노트북은 Corp/src/04_A2_코넥스_선별에서 실행)
sys.path.insert(0, str(BASE / "src" / "01_파일럿"))
import config                                         # SEED 같은 설정값을 가져옴

KIND = BASE / "outputs" / "01_파일럿" / "KIND_상장폐지목록_2019_2024_명단대조.csv"
COMPANY = BASE / "outputs" / "01_파일럿" / "기업개황_상장사전체.csv"
OUTDIR = BASE / "outputs" / "02_코넥스_선별"
SAVE_FILES = True            # 새 파일을 만드는 노트북이라 기본값 True(기존 파일이 있으면 덮어씀)

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 150)
print("Corp 폴더:", BASE.name)

Corp 폴더: Corp


---
# 3. 조회 대상 만들기

**코넥스 기업**은 세 가지 경로로 모읍니다(제조업 = 업종 앞 2자리 10~34, 12월 결산만).

| 출처 | 뜻 |
|---|---|
| **현재 코넥스** | 지금 법인구분이 N(코넥스)인 제조업 기업 |
| **KIND 폐지(코넥스)** | KIND에서 폐지될 당시 시장이 코넥스였고, 우리 명단에서 이름이 확인된 제조업 기업 |
| **KIND 시장 이전(코넥스)** | 코넥스에서 코스닥 등으로 옮겨 간 제조업 기업 (사전기간에는 코넥스에 있었음) |

**비교 기업**은 지금 코스닥(K)인 제조업 중 12월 결산 기업에서 **무작위 40곳**(시드 고정)입니다. 지금 살아 있는 기업이라 코넥스 쪽보다 유리한 비교입니다. 판정에는 쓰지 않고 참고만 합니다.

In [2]:
k = pd.read_csv(KIND, dtype=str, keep_default_na=False)
c = pd.read_csv(COMPANY, dtype=str, keep_default_na=False)
is_manu = c["induty_code"].str[:2].between("10", "34")

# (1) 현재 코넥스
cur = c[(c["corp_cls"] == "N") & is_manu][["corp_code"]].assign(출처="현재 코넥스", 동명모호="")

# (2)(3) KIND에서 폐지·이전된 코넥스 제조업
kx = k[(k["시장구분"] == "코넥스") & (k["일치여부"] == "Y") & (k["제조업(앞2자리10~34)"] == "Y") & (~k["분류"].str.startswith("스팩"))].copy()
kx["출처"] = kx["분류"].map(lambda s: "KIND 시장 이전(코넥스)" if s.startswith("시장") else "KIND 폐지(코넥스)")
kx["동명모호"] = kx["DART명단_일치"].map(lambda s: "Y" if "동명" in s else "")
kx = kx[["corp_code", "출처", "동명모호"]]

both = pd.concat([cur, kx], ignore_index=True)
grp = both.groupby("corp_code").agg(출처=("출처", " + ".join), 동명모호=("동명모호", lambda s: "Y" if (s == "Y").any() else "")).reset_index()
grp = grp.merge(c[["corp_code", "corp_name", "corp_cls", "induty_code", "acc_mt"]], on="corp_code", how="left")
grp = grp[grp["acc_mt"] == "12"].copy()                 # 12월 결산만(A2 제안과 같은 기준)
grp["집단"] = "코넥스"

# 비교: 현재 코스닥 제조업, 12월 결산 중 무작위 40곳
pool = c[(c["corp_cls"] == "K") & is_manu & (c["acc_mt"] == "12")]["corp_code"].tolist()
pick = random.Random(config.SEED).sample(pool, 40)
base = c[c["corp_code"].isin(pick)][["corp_code", "corp_name", "corp_cls", "induty_code", "acc_mt"]].assign(출처="현재 코스닥(무작위)", 동명모호="", 집단="비교_코스닥")

target = pd.concat([grp, base], ignore_index=True)[["집단", "corp_code", "corp_name", "출처", "동명모호", "corp_cls", "induty_code", "acc_mt"]]
target = target.rename(columns={"corp_name": "회사명(DART)", "corp_cls": "현재_법인구분", "induty_code": "업종코드", "acc_mt": "결산월"})
print("조회 대상:", len(target), "곳")
print(target["집단"].value_counts().to_dict())

조회 대상: 157 곳
{'코넥스': 117, '비교_코스닥': 40}


---
# 4. 대상 확인

In [3]:
print("[코넥스 집단의 출처]")
print(target[target["집단"] == "코넥스"]["출처"].value_counts().to_string())
print()
print("동명이라 같은 회사인지 모호한 기업:", int((target["동명모호"] == "Y").sum()), "곳")
print("코넥스 집단의 현재 법인구분:", target[target["집단"] == "코넥스"]["현재_법인구분"].value_counts().to_dict())
target[target["집단"] == "코넥스"].head(10)

[코넥스 집단의 출처]
출처
현재 코넥스             62
KIND 폐지(코넥스)       34
KIND 시장 이전(코넥스)    21

동명이라 같은 회사인지 모호한 기업: 7 곳
코넥스 집단의 현재 법인구분: {'N': 62, 'E': 30, 'K': 25}


,집단,corp_code,회사명(DART),출처,동명모호,현재_법인구분,업종코드,결산월
0,코넥스,00311216,에이치엔에스하이텍,KIND 시장 이전(코넥스),,K,262,12
1,코넥스,00341314,엔더블유시,현재 코넥스,,N,264,12
2,코넥스,00393618,제노텍,현재 코넥스,,N,204,12
3,코넥스,00407036,선바이오,KIND 시장 이전(코넥스),,K,212,12
4,코넥스,00414540,대주이엔티,현재 코넥스,,N,2413,12
5,코넥스,00442631,태양기계,현재 코넥스,,N,30320,12
6,코넥스,00474588,한중엔시에스,KIND 시장 이전(코넥스),,K,281,12
7,코넥스,00521433,엔지브이아이,현재 코넥스,,N,30399,12
8,코넥스,00523176,광동헬스바이오,현재 코넥스,,N,108,12
9,코넥스,00533809,바스칸바이오제약,현재 코넥스,,N,212,12


### 해석
- **현재 코넥스**는 지금도 코넥스에 있는 기업, **KIND 폐지**는 코넥스에서 사라진 기업, **KIND 시장 이전**은 코스닥 등으로 옮긴 기업입니다. 한 기업이 두 출처에 걸리면 `+`로 합쳐 표시됩니다.
- **KIND에서 온 기업의 현재 법인구분은 대부분 E 또는 K**입니다(폐지 → E, 이전 → K).
- **동명 모호 기업**은 같은 이름의 회사가 우리 명단에 둘 이상이라, 이름 대조로 잡은 기업이 다른 회사일 수 있습니다. 포함은 하되 표시해 두었고, 판정에서는 이들을 뺀 결과도 같이 봅니다.
- 이 명단은 **KIND 목록에 있고 우리 명단에서 이름이 확인된 기업과 현재 코넥스 기업**입니다. 이름이 확인되지 않은 코넥스 폐지 기업(미확인 54건 중 일부)은 들어 있지 않습니다. 따라서 이 명단은 **코넥스 기업 전체가 아닙니다.**

---
# 5. 호출 수와 실행 방법

In [4]:
NEED = [(2019, "Q1"), (2019, "H1"), (2019, "Q3"), (2019, "FY"), (2020, "Q1"), (2020, "H1"), (2020, "Q3"), (2020, "FY"), (2021, "Q1"), (2021, "H1")]
firms = len(target)
print(f"기업 {firms}곳 × 보고서 {len(NEED)}개 = {firms*len(NEED):,}건 (연결이 비어 별도로 다시 묻는 경우까지 최대 {firms*len(NEED)*2:,}건)")
print(f"호출 사이 대기 {config.SLEEP}초 → 대기 시간만 최소 약 {firms*len(NEED)*config.SLEEP/60:.0f}분 (서버 응답 시간 제외)")

# 이미 받아 둔 보고서(파일럿)는 다시 요청하지 않는다
fs_dir = config.RAW / "fs"
cached = 0
for r in target.itertuples():
    for y, rep in NEED:
        code = config.REPORTS[rep]
        for fs in ("CFS", "OFS"):
            if (fs_dir / f"{r.corp_code}_{y}_{code}_{fs}.json").exists():
                cached += 1
print("이미 저장된 응답 파일 수(파일럿에서 받은 것):", cached)

기업 157곳 × 보고서 10개 = 1,570건 (연결이 비어 별도로 다시 묻는 경우까지 최대 3,140건)
호출 사이 대기 0.4초 → 대기 시간만 최소 약 10분 (서버 응답 시간 제외)


이미 저장된 응답 파일 수(파일럿에서 받은 것):

 80


In [5]:
if SAVE_FILES:
    OUTDIR.mkdir(exist_ok=True)
    target.to_csv(OUTDIR / "코넥스_선별대상_제조업.csv", index=False, encoding="utf-8-sig")
    print("저장:", "outputs/02_코넥스_선별/코넥스_선별대상_제조업.csv", f"({len(target)}행)")
else:
    print("SAVE_FILES=False → 저장하지 않았습니다.")

저장: outputs/02_코넥스_선별/코넥스_선별대상_제조업.csv (157행)


### 실행 방법 (사용자가 직접)

DART 호출에는 인증키가 필요하고, 키는 **본인 PowerShell 창**에서만 설정합니다(키를 채팅·파일에 적지 않습니다).

```powershell
cd "…\Corp\src\04_A2_코넥스_선별"
$env:DART_API_KEY = "본인 인증키"
python -X utf8 02_코넥스_선별_수집.py --dry-run     # 먼저: 호출 수만 계산(요청 안 보냄)
python -X utf8 02_코넥스_선별_수집.py               # 실제 조회(약 20~40분)
```

- **중간에 멈춰도 됩니다.** 받은 응답은 저장되어 있어서 같은 명령을 다시 실행하면 이어서 합니다. 요청 제한(020)이 나오면 스크립트가 그 자리에서 멈춥니다. 이때도 내일 같은 명령으로 이어서 하면 됩니다.
- 끝나면 화면에 **M, 비교 기준값, 규칙 판정**이 출력되고 결과 csv 두 개가 `outputs/02_코넥스_선별/`에 저장됩니다.
- 결과를 보내 주시면 해석 노트북을 만들겠습니다.

---
# 6. 한계

| 한계 | 이유 |
|---|---|
| 대상이 코넥스 기업 전체가 아님 | KIND 대조에서 이름이 확인된 기업과 현재 코넥스 기업만 포함 |
| 50%, 20곳은 임의 기준 | 공식·학술 기준 없음. 결과를 보기 전에 적어 두는 것이 이 규칙의 의미 |
| 비교 기업은 지금 살아 있는 코스닥 기업 | 폐지된 코스닥 기업은 들어 있지 않아 코스닥 쪽이 유리함 |
| "보고서가 있다"는 DART API가 응답을 준다는 뜻 | 기업이 실제로 제출했는지와 API 응답이 다를 수 있음(예: 일부 기업 제공 범위) |
| 코넥스 기업의 분기 보고서 제출 의무 | 제도를 이번에 확인하지 않았음. 조회 결과가 "제출 안 함"인지 "API가 안 줌"인지 구분 못 함 |